# DSA 504 — Class 10
## Programmatic Data Access I: File Formats

**Date:** Monday, Oct 5
**Reading:** *Python for Data Analysis*, ch. 6

---

Every class so far has used one file format: CSV. Real data doesn't always show up that way — you'll also encounter JSON (common from web APIs), Excel (common in business settings), and Parquet (common in larger-scale data engineering). Today covers reading and writing all of them.

### Learning goals
By the end of this class, you will be able to:
- Read and write CSV files with more control (custom separators, encoding, missing-value markers)
- Read and write JSON, and understand how it relates to the list-of-dictionaries pattern from Class 3
- Read and write Excel files, including multiple sheets
- Read and write Parquet files, and explain why they're often preferred for larger datasets
- Choose an appropriate file format for a given situation
- Read a large file in chunks, for cases where it doesn't fit comfortably in memory


In [1]:
import pandas as pd
import numpy as np
import os

sales = pd.read_csv("retail_sales_clean.csv")
sales["date"] = pd.to_datetime(sales["date"])
sales.head()


,date,store,units_sold,revenue,category,is_end_of_month
0,2025-04-22,Rome,25.0,402.50,toys,False
1,2025-04-12,Syracuse,38.0,3154.00,electronics,False
2,2025-12-21,Utica,42.0,1299.06,home goods,False
3,2025-02-11,Syracuse,60.0,5114.40,electronics,False
4,2025-04-10,Syracuse,10.0,147.10,toys,False


## 1. CSV — A Closer Look

You've used `pd.read_csv()` and (in HW exercises) `.to_csv()` since Class 4. Here are a few options worth knowing that we haven't needed yet.


In [ ]:
%pwd

In [ ]:
# Writing without the index column (you've done this before, worth re-confirming why)
sales.to_csv("sales_export.csv", index=False)

# Reading back, and controlling how missing values are recognized on the way in
sales_reloaded = pd.read_csv("sales_export.csv", na_values=["", "NA", "N/A", "null"])
print(sales_reloaded.shape)


**Why `index=False` matters:** without it, pandas writes its own row-number index as an extra unnamed column in the file. That's rarely what you want — it's not real data, just pandas' internal bookkeeping, and it clutters the file for anyone else who opens it.

**Why `na_values` matters:** different systems mark "missing" differently — some use blank cells, others literally write the text `"NA"` or `"null"`. Telling pandas up front which of these to treat as missing prevents them from silently being read in as plain text strings.


## 2. JSON

**JSON** (JavaScript Object Notation) is a text format built around exactly the key-value structure you already know from Class 3 dictionaries. It's the most common format for data returned by web APIs (which we cover next class).


In [ ]:
# Writing a DataFrame to JSON
sales.head(3).to_json("sales_sample.json", orient="records", indent=2, date_format="iso")

# Let's look at the raw text to see the structure
with open("sales_sample.json") as f:
    print(f.read())


**Look closely at that structure.** It's a list of dictionaries — `[{...}, {...}, {...}]` — exactly the pattern from Class 3, just written as text instead of live Python objects. This is not a coincidence: JSON *is* basically Python's list-of-dicts pattern, standardized so any programming language (not just Python) can read and write it.


In [ ]:
# Reading JSON back into a DataFrame
reloaded = pd.read_json("sales_sample.json")
reloaded


### The `orient` parameter

JSON can represent a table in more than one shape. `orient="records"` (used above) gives one dictionary per row — usually the most intuitive shape. Other options exist (`"columns"`, `"index"`, `"split"`), but `"records"` covers the large majority of real-world cases you'll encounter.


In [ ]:
# orient="columns" -- a very different shape: one dictionary per COLUMN, not per row
sales.head(3).to_json("sales_sample_columns.json", orient="columns", indent=2, date_format="iso")
with open("sales_sample_columns.json") as f:
    print(f.read())


## 3. Excel

Despite everything else in this course, Excel remains extremely common in real workplaces — many non-technical colleagues will expect data in this format, not CSV or JSON.


In [ ]:
# Writing to Excel (requires the openpyxl package, already installed for this course)
sales.head(20).to_excel("sales_sample.xlsx", index=False, sheet_name="Sales")

# Reading it back
reloaded_excel = pd.read_excel("sales_sample.xlsx", sheet_name="Sales")
reloaded_excel.head()


### Multiple sheets in one file

A single Excel *file* (a "workbook") can contain multiple *sheets* — this is genuinely different from CSV, which only ever has one table per file.


In [ ]:
# Writing multiple sheets to the same Excel file using ExcelWriter
revenue_by_store = sales.groupby("store")["revenue"].sum().reset_index()
revenue_by_category = sales.groupby("category")["revenue"].sum().reset_index()

with pd.ExcelWriter("sales_summary.xlsx") as writer:
    sales.head(20).to_excel(writer, sheet_name="Raw Data", index=False)
    revenue_by_store.to_excel(writer, sheet_name="By Store", index=False)
    revenue_by_category.to_excel(writer, sheet_name="By Category", index=False)

print("Saved 3 sheets to sales_summary.xlsx")


In [ ]:
# Reading a SPECIFIC sheet back
by_store = pd.read_excel("sales_summary.xlsx", sheet_name="By Store")
print(by_store)

# Reading ALL sheets at once returns a dictionary of DataFrames, keyed by sheet name
all_sheets = pd.read_excel("sales_summary.xlsx", sheet_name=None)
print("\nSheet names found:", list(all_sheets.keys()))


## 4. Parquet

**Parquet** is a binary (not plain-text) file format built specifically for data analysis. Unlike CSV or JSON, you can't open it and read it directly — but it has real advantages that matter once datasets get larger than our small teaching dataset.


In [ ]:
%pip install pyarrow openpyxl

In [ ]:
# Writing and reading Parquet (requires the pyarrow package, already installed)
sales.to_parquet("sales.parquet", index=False)

reloaded_parquet = pd.read_parquet("sales.parquet")
print(reloaded_parquet.dtypes)
reloaded_parquet.head()


**Why Parquet matters — comparing file sizes:**


In [ ]:
# Comparing file sizes across formats, for the SAME data
sales.to_csv("compare.csv", index=False)
sales.to_json("compare.json", orient="records", date_format="iso")
sales.to_parquet("compare.parquet", index=False)

for filename in ["compare.csv", "compare.json", "compare.parquet"]:
    size_kb = os.path.getsize(filename) / 1024
    print(f"{filename}: {size_kb:.1f} KB")


**Parquet is usually noticeably smaller than CSV or JSON for the same data**, because it's compressed and stores each column efficiently based on its actual type. It also **preserves data types exactly** — reload a Parquet file and `date` is still a real datetime, no `pd.to_datetime()` needed, unlike CSV which always round-trips through plain text. The tradeoff: you can't open it in a text editor or Excel directly, and older tools may not support it. For this course's dataset size, CSV is perfectly fine — but for real datasets with millions of rows, Parquet is often the better default.


## 5. Choosing the Right Format

| Situation | Best format |
|---|---|
| Sharing with non-technical colleagues, or opening manually to inspect | CSV or Excel |
| Data from or to a web API | JSON |
| A workbook needs multiple related tables together | Excel (multiple sheets) |
| Large datasets, or you need exact dtypes preserved | Parquet |
| Quick, universal compatibility with almost any tool | CSV |


## 6. Reading Large Files in Chunks

Our dataset is small enough to load entirely into memory at once. Real datasets are sometimes too large for that. `pd.read_csv()` supports a `chunksize` parameter, which reads the file in pieces instead of all at once.


In [ ]:
# Reading in chunks of 500 rows at a time, and processing each chunk separately
total_revenue = 0
chunk_count = 0

for chunk in pd.read_csv("retail_sales_clean.csv", chunksize=500):
    # print(f"Processing chunk {chunk_count + 1} with {len(chunk)} rows")
    total_revenue += chunk["revenue"].sum()
    chunk_count += 1

print(f"Processed {chunk_count} chunks")
print(f"Total revenue (computed via chunks): ${total_revenue:,.2f}")

# Confirming this matches loading it all at once
print(f"Total revenue (all at once): ${sales['revenue'].sum():,.2f}")


**Why this matters:** if a file is too large to fit in your computer's memory at once, loading it normally will crash or freeze your program. Processing it in chunks — reading a piece, doing your calculation, discarding that piece, reading the next — solves that, at the cost of writing slightly more careful code (as above, keeping a running total instead of one big DataFrame in memory).


---
## Guided Practice

Work through these using `sales`, already loaded above.


### Exercise 1 — CSV round-trip
Save just the `store` and `revenue` columns to a new CSV file called `store_revenue.csv`, without the index column. Read it back in and confirm the shape matches what you expect.


In [ ]:
# Exercise 1 — your code here



### Exercise 2 — JSON
Save the first 5 rows of `sales` to a JSON file using `orient="records"`. Open the file with plain Python (`open()` and `.read()`, not pandas) and print its contents to confirm the list-of-dictionaries structure.


In [ ]:
# Exercise 2 — your code here



### Exercise 3 — Excel with multiple sheets
Create an Excel file with two sheets: one containing total units sold per store, and another containing total units sold per category. Then read the file back and print the names of the sheets found.


In [ ]:
# Exercise 3 — your code here



### Exercise 4 — Parquet and file size
Save `sales` to both CSV and Parquet. Print the file size of each in KB, and state which one is smaller.


In [ ]:
# Exercise 4 — your code here



### Exercise 5 (stretch) — Chunked processing
Using `chunksize=1000`, read `retail_sales_clean.csv` in chunks and compute the total number of rows where `revenue` is greater than 3000, without ever loading the whole file into memory at once.


In [ ]:
# Exercise 5 — your code here



---
## Solutions

Try each exercise yourself first. These are here for after class, or once you're stuck.


---
## Wrap-up

**Recap:** finer control over CSV reading/writing (`index=False`, `na_values`); JSON as the text form of the list-of-dictionaries pattern, and the `orient` parameter; Excel files and multi-sheet workbooks; Parquet's size and dtype-preservation advantages over CSV; a format-selection guide; and reading large files in chunks with `chunksize`.

**Common mistakes to watch for:**
- Forgetting `index=False` when writing to CSV or Excel, ending up with a stray unnamed column
- Assuming a Parquet file can be opened and read like a text file — it can't; you need pandas (or a similar tool) to read it
- Trying to build up one giant DataFrame while processing chunks, instead of accumulating just the result you need (defeats the whole purpose of chunking)

**No homework due today. HW3 is assigned at the end of Class 11.**

**Before next class (Oct 7):** Class 11 covers the other major way real data arrives — calling a web API — plus handling credentials (like API keys) safely, without hard-coding them into your script.
